# Chapter 6 Video Walkthrough: Single-Leaf Taproot (Offline Deterministic)

This notebook is optimized for teaching + recording.

## What this walkthrough covers

- Commit phase: build a single-leaf hash-lock Taproot address.
- Reveal phase A: key path spending (minimal witness, highest privacy).
- Reveal phase B: script path spending (preimage path, full unlock flow).
- Funding narrative: two separate UTXOs are used (one for key path, one for script path).
- API mapping: `bitcoinutils` low-level flow vs `btcaaron` high-level flow.

## Execution mode

- Offline deterministic by default.
- No mandatory broadcast calls.
- Optional high-level track is guarded (safe to skip if env is missing deps).

In [24]:
# Environment check
import sys
import hashlib
import struct
from pathlib import Path

from bitcoinutils.setup import setup
from bitcoinutils.keys import PrivateKey, P2trAddress
from bitcoinutils.script import Script
from bitcoinutils.transactions import Transaction, TxInput, TxOutput, TxWitnessInput
from bitcoinutils.utils import to_satoshis, ControlBlock

setup('testnet')

print('Python:', sys.version.split()[0])
print('bitcoinutils imports: OK')
print('Working directory:', Path.cwd())

Python: 3.11.11
bitcoinutils imports: OK
Working directory: /Volumes/MAC_Programs/code401/codes_401/mastering-taproot/notebooks


## Concept map (for coding-only session)

- Internal key `P` + single leaf script -> Taproot output key `Q`.
- Commit phase: lock to one Taproot address (looks like a normal payment).
- Reveal phase (choose one):
  - Key path witness: `[signature]`
  - Script path witness: `[preimage, script, control_block]`
- In this Chapter 6 coding session, we focus on full flow and transaction assembly.

In [18]:
# Shared deterministic setup
ALICE_WIF = 'cRxebG1hY6vVgS9CSLNaEbEJaXkpZvc6nFeqqGT7v6gcW7MbzKNT'
PREIMAGE = 'helloworld'


def tagged_hash(tag: str, data: bytes) -> bytes:
    tag_hash = hashlib.sha256(tag.encode()).digest()
    return hashlib.sha256(tag_hash + tag_hash + data).digest()


def build_hash_lock_script(preimage: str) -> Script:
    preimage_hash = hashlib.sha256(preimage.encode('utf-8')).hexdigest()
    return Script(['OP_SHA256', preimage_hash, 'OP_EQUALVERIFY', 'OP_TRUE'])


alice_private = PrivateKey(ALICE_WIF)
alice_public = alice_private.get_public_key()
tr_script = build_hash_lock_script(PREIMAGE)

print('Alice x-only:', alice_public.to_x_only_hex())
print('Hash-lock script:', tr_script.to_hex())
print('Preimage hash:', hashlib.sha256(PREIMAGE.encode()).hexdigest())

Alice x-only: 50be5fc44ec580c387bf45df275aaa8b27e2d7716af31f10eeed357d126bb4d3
Hash-lock script: a820936a185caaa266bb9cbe981e9e05cb78cd732b0b3280eb944412bb6f8f8f07af8851
Preimage hash: 936a185caaa266bb9cbe981e9e05cb78cd732b0b3280eb944412bb6f8f8f07af


## Track A (`bitcoinutils`) — Step 1: Commit phase address construction

In [25]:
commit_address = alice_public.get_taproot_address([[tr_script]])
print('Commit address:', commit_address.to_string())
print('ScriptPubKey: ', commit_address.to_script_pub_key().to_hex())
print('Parity is_odd:', commit_address.is_odd())

Commit address: tb1p53ncq9ytax924ps66z6al3wfhy6a29w8h6xfu27xem06t98zkmvsakd43h
ScriptPubKey:  5120a46780148be98aaa861ad0b5dfc5c9b935d515c7be8c9e2bc6cedfa594e2b6d9
Parity is_odd: True


## Track A — Step 2: Funding UTXO #1 -> Reveal phase (key path)

Context for video:
- We already funded the commit address once.
- `key_prev_txid` is that funding transaction (UTXO #1).
- Now we unlock UTXO #1 through key path.

In [20]:
# Funding UTXO #1 (already committed in the chapter flow)
key_prev_txid = '4fd83128fb2df7cd25d96fdb6ed9bea26de755f212e37c3aa017641d3d2d2c6d'
key_input_amount = 0.00003900
key_output_amount = 0.00003700
shared_output_addr = P2trAddress('tb1p060z97qusuxe7w6h8z0l9kam5kn76jur22ecel75wjlmnkpxtnls6vdgne')

key_txin = TxInput(key_prev_txid, 0)
key_txin.sequence = struct.pack('<I', 0xffffffff)
key_txout = TxOutput(to_satoshis(key_output_amount), shared_output_addr.to_script_pub_key())
key_tx = Transaction([key_txin], [key_txout], has_segwit=True)

key_sig = alice_private.sign_taproot_input(
    key_tx,
    0,
    [commit_address.to_script_pub_key()],
    [to_satoshis(key_input_amount)],
    script_path=False,
    tapleaf_scripts=[tr_script],
)
key_tx.witnesses.append(TxWitnessInput([key_sig]))

print('UTXO #1 funding txid (input):', key_prev_txid)
print('UTXO #1 funding amount       :', to_satoshis(key_input_amount), 'sats')
print('Key path spend txid          :', key_tx.get_txid())
print('Key path witness             : [signature]')
print('Witness[0] signature         :', key_sig[:16] + '...' + key_sig[-16:])

UTXO #1 funding txid (input): 4fd83128fb2df7cd25d96fdb6ed9bea26de755f212e37c3aa017641d3d2d2c6d
UTXO #1 funding amount       : 3900 sats
Key path spend txid          : 85e843d5fd6273d2668cbaa787be4bed918b4dac4dba4d305c8cc1f4618b9af1
Key path witness             : [signature]
Witness[0] signature         : 4cbfeda0f020c883...c3726469f69411a2


## Track A — Step 3: Funding UTXO #2 -> Reveal phase (script path)

Context for video:
- After the key-path example, we fund the same commit address again.
- `script_prev_txid` is that second funding transaction (UTXO #2).
- Now we unlock UTXO #2 through script path.

In [26]:
# Funding UTXO #2 (second commit funding)
script_prev_txid = '9e193d8c5b4ff4ad7cb13d196c2ecc210d9b0ec144bb919ac4314c1240629886'
script_input_amount = 0.00005000
script_output_amount = 0.00004000

script_txin = TxInput(script_prev_txid, 0)
script_txin.sequence = struct.pack('<I', 0xffffffff)
script_txout = TxOutput(to_satoshis(script_output_amount), shared_output_addr.to_script_pub_key())
script_tx = Transaction([script_txin], [script_txout], has_segwit=True)

control_block = ControlBlock(
    alice_public,
    [[tr_script]],
    0,
    is_odd=commit_address.is_odd(),
)

preimage_hex = PREIMAGE.encode('utf-8').hex()
script_witness_items = [preimage_hex, tr_script.to_hex(), control_block.to_hex()]
script_tx.witnesses.append(TxWitnessInput(script_witness_items))

print('UTXO #2 funding txid (input):', script_prev_txid)
print('UTXO #2 funding amount       :', to_satoshis(script_input_amount), 'sats')
print('Script path spend txid       :', script_tx.get_txid())
print('Script path witness layout   : [preimage, script, control_block]')
print('Witness[0] preimage          :', script_witness_items[0])
print('Witness[1] script            :', script_witness_items[1][:16] + '...' + script_witness_items[1][-16:])

UTXO #2 funding txid (input): 9e193d8c5b4ff4ad7cb13d196c2ecc210d9b0ec144bb919ac4314c1240629886
UTXO #2 funding amount       : 5000 sats
Script path spend txid       : 68f7c8f0ab6b3c6f7eb037e36051ea3893b668c26ea6e52094ba01a7722e604f
Script path witness layout   : [preimage, script, control_block]
Witness[0] preimage          : 68656c6c6f776f726c64
Witness[1] script            : a820936a185caaa2...bb6f8f8f07af8851


## Track B (`btcaaron`) — Step 1: Build same commit address + key-path spend

This part mirrors Track A with high-level API.

- Use one UTXO for key-path spend first.
- Keep output deterministic for recording.

In [27]:
btcaaron_repo = Path('/Volumes/MAC_Programs/code401/codes_401/btcaaron')
if btcaaron_repo.exists() and str(btcaaron_repo) not in sys.path:
    sys.path.insert(0, str(btcaaron_repo))

has_btcaaron = False
try:
    from btcaaron import Key, TapTree
    has_btcaaron = True
except Exception as e:
    print('btcaaron import skipped:', e)

if has_btcaaron:
    alice_key = Key.from_wif(ALICE_WIF)
    program = TapTree(internal_key=alice_key).hashlock(PREIMAGE, label='hash').build()

    key_tx_b = (
        program.keypath()
        .from_utxo(key_prev_txid, 0, sats=to_satoshis(key_input_amount))
        .to(shared_output_addr.to_string(), to_satoshis(key_output_amount))
        .sign(alice_key)
        .sequence(0xffffffff)
        .build()
    )

    print('btcaaron commit address      :', program.address)
    print('bitcoinutils commit address  :', commit_address.to_string())
    print('Address match                :', program.address == commit_address.to_string())
    print('btcaaron key-path spend txid :', key_tx_b.txid)
else:
    print('Skip Track B. Install missing dependencies before recording full Track B.')

btcaaron commit address      : tb1p53ncq9ytax924ps66z6al3wfhy6a29w8h6xfu27xem06t98zkmvsakd43h
bitcoinutils commit address  : tb1p53ncq9ytax924ps66z6al3wfhy6a29w8h6xfu27xem06t98zkmvsakd43h
Address match                : True
btcaaron key-path spend txid : 85e843d5fd6273d2668cbaa787be4bed918b4dac4dba4d305c8cc1f4618b9af1


In [15]:
## Track B (`btcaaron`) — Step 2: Script-path spend

In [28]:
if has_btcaaron:
    script_tx_b = (
        program.spend('hash')
        .from_utxo(script_prev_txid, 0, sats=to_satoshis(script_input_amount))
        .to(shared_output_addr.to_string(), to_satoshis(script_output_amount))
        .unlock(preimage=PREIMAGE)
        .sequence(0xffffffff)
        .build()
    )

    print('btcaaron script-path spend txid:', script_tx_b.txid)
    print('Track A script-path spend txid :', script_tx.get_txid())
    print('Same input txid used in Track B:', script_prev_txid)
else:
    print('Skip Track B script-path spend.')

btcaaron script-path spend txid: 68f7c8f0ab6b3c6f7eb037e36051ea3893b668c26ea6e52094ba01a7722e604f
Track A script-path spend txid : 68f7c8f0ab6b3c6f7eb037e36051ea3893b668c26ea6e52094ba01a7722e604f
Same input txid used in Track B: 9e193d8c5b4ff4ad7cb13d196c2ecc210d9b0ec144bb919ac4314c1240629886


## Final TXID check

In [17]:
print('Track A key-path txid   :', key_tx.get_txid())
print('Track A script-path txid:', script_tx.get_txid())
if has_btcaaron:
    print('Track B key-path txid   :', key_tx_b.txid)
    print('Track B script-path txid:', script_tx_b.txid)

Track A key-path txid   : 85e843d5fd6273d2668cbaa787be4bed918b4dac4dba4d305c8cc1f4618b9af1
Track A script-path txid: 68f7c8f0ab6b3c6f7eb037e36051ea3893b668c26ea6e52094ba01a7722e604f
Track B key-path txid   : 85e843d5fd6273d2668cbaa787be4bed918b4dac4dba4d305c8cc1f4618b9af1
Track B script-path txid: 68f7c8f0ab6b3c6f7eb037e36051ea3893b668c26ea6e52094ba01a7722e604f


<div></div>

In [ ]:
if not has_btcaaron:
    print('Tip: optional -> pip install requests')

<div></div>